# IBM Quantum hardware audit (optional, Open Plan)

This notebook checks four **4-qubit ZZ feature-map fidelities** on a real IBM QPU and compares them with exact noiseless values. It does **not** train a full hardware QSVM or modify the existing ML–QML leaderboards. A full kernel requires too many circuit evaluations for this small free-plan audit.

Run All is safe by default: it performs local circuit checks only. Hardware submission is disabled until you deliberately enable it below. Never paste an API key into a notebook cell or commit it to Git. IBM's [Open Plan](https://quantum.cloud.ibm.com/docs/en/guides/plans-overview) currently offers up to 10 QPU minutes per 28-day rolling window. Open Plan jobs use [job or batch mode, not sessions](https://quantum.cloud.ibm.com/docs/en/guides/run-jobs-session). Check the IBM dashboard for your current allowance before submission.


## 1. Settings and dependencies

Use the project Python kernel. `qiskit-ibm-runtime` is required only for the account and hardware cells; if it is missing, install the project's `requirements-stage4.txt` into that kernel environment.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import Statevector
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'notebooks'/'09_equal_sample_benchmark.ipynb').exists()
OPEN_INSTANCE_CRN = None  # Paste ONLY the CRN of an IBM Quantum Open Plan instance
SAVE_OPEN_ACCOUNT = False  # One-time secure prompt; no token is placed in notebook source
CONNECT_TO_IBM = False  # Lists a QPU, but does not submit a job
SUBMIT_NEW_JOB = False  # Requires confirmation below; consumes Open Plan allowance
CONFIRM_OPEN_PLAN = False  # Set True only after checking this CRN is Open Plan
ALLOW_RESUBMIT = False  # Protects against accidental duplicate jobs
SHOTS = 256
assert 1 <= SHOTS <= 512, 'Keep this audit small for free-plan use.'
OUT = ROOT/'results'/'ibm_hardware_audit'
print('Local checks ready. Hardware submission is disabled:', not SUBMIT_NEW_JOB)

## 2. Build and test four fidelity circuits locally

The probability of measuring `0000` after applying `U(a)` then `U(b)†` estimates `|⟨φ(b)|φ(a)⟩|²`. The first pair uses identical inputs as a sanity check. Local statevector results should match before any hardware job is submitted.


In [ ]:
def fidelity_circuit(a, b):
    feature_map = zz_feature_map(4, reps=1, entanglement='linear')
    circuit = QuantumCircuit(4, 4)
    circuit.compose(feature_map.assign_parameters(a), inplace=True)
    circuit.compose(feature_map.assign_parameters(b).inverse(), inplace=True)
    circuit.measure(range(4), range(4))
    return circuit

def ideal_fidelity(a, b):
    feature_map = zz_feature_map(4, reps=1, entanglement='linear')
    va = Statevector.from_instruction(feature_map.assign_parameters(a)).data
    vb = Statevector.from_instruction(feature_map.assign_parameters(b)).data
    return float(abs(np.vdot(va, vb))**2)

base = np.random.default_rng(42).uniform(-1, 1, size=4)
vectors = np.array([base, base+0.03, base+0.07, base+0.12])
pairs = [(0,0), (0,1), (0,2), (0,3)]
circuits = [fidelity_circuit(vectors[i], vectors[j]) for i,j in pairs]
ideals = [ideal_fidelity(vectors[i], vectors[j]) for i,j in pairs]
for circuit, ideal in zip(circuits, ideals):
    unmeasured = circuit.remove_final_measurements(inplace=False)
    zero_probability = Statevector.from_instruction(unmeasured).probabilities_dict().get('0000', 0.0)
    assert np.isclose(zero_probability, ideal, atol=1e-12)
assert np.isclose(ideals[0], 1.0, atol=1e-12)
display(pd.DataFrame({'pair':pairs, 'ideal_fidelity':ideals}))
print('Local circuit checks passed; no QPU time used.')

## 3. Save IBM credentials once, securely (optional)

Create an [IBM Quantum Platform account and Open Plan instance](https://quantum.cloud.ibm.com/docs/en/guides/cloud-setup). Copy its **Open Plan CRN** from the Instances page. Create an API key in the dashboard. Set `OPEN_INSTANCE_CRN` and `SAVE_OPEN_ACCOUNT=True` above; this cell prompts invisibly for the key and stores credentials in Qiskit's user configuration, outside this repository. Then return `SAVE_OPEN_ACCOUNT` to `False`. Do not share your key or commit Qiskit's credential file. [IBM credential guide](https://quantum.cloud.ibm.com/docs/en/guides/save-credentials).


In [ ]:
if SAVE_OPEN_ACCOUNT:
    assert OPEN_INSTANCE_CRN and OPEN_INSTANCE_CRN.startswith('crn:'), 'Enter your Open Plan instance CRN above.'
    from getpass import getpass
    from qiskit_ibm_runtime import QiskitRuntimeService
    QiskitRuntimeService.save_account(
        channel='ibm_quantum_platform', token=getpass('IBM Cloud API key: '),
        instance=OPEN_INSTANCE_CRN, name='ctrlq_open', set_as_default=False, overwrite=False,
    )
    print('Saved account locally; key was not printed or written into the notebook.')
else:
    print('Credential setup skipped.')

## 4. Inspect an available QPU (no job yet)

Set `CONNECT_TO_IBM=True` only after saving credentials and confirming the CRN is an Open Plan instance. The QPU may change between preview and submission.


In [ ]:
service = None
backend = None
isa_circuits = None
if CONNECT_TO_IBM:
    assert OPEN_INSTANCE_CRN and OPEN_INSTANCE_CRN.startswith('crn:') and CONFIRM_OPEN_PLAN
    from qiskit_ibm_runtime import QiskitRuntimeService
    from qiskit.transpiler import generate_preset_pass_manager
    service = QiskitRuntimeService(channel='ibm_quantum_platform', instance=OPEN_INSTANCE_CRN)
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=4)
    isa_circuits = generate_preset_pass_manager(backend=backend, optimization_level=1).run(circuits)
    assert len(isa_circuits) == len(circuits)
    print('Backend:', backend.name, '| circuits:', len(isa_circuits), '| shots each:', SHOTS)
    print('Review your Open Plan usage in IBM Quantum Platform before submission.')
else:
    print('IBM connection skipped; no QPU time used.')

## 5. Submit one small job (explicit opt-in)

Set `SUBMIT_NEW_JOB=True` only when ready. This uses **job mode**, not a session. The job ID is saved immediately so it can be recovered if Jupyter closes. If a prior job file exists, submission stops unless you deliberately set `ALLOW_RESUBMIT=True`. Do not keep rerunning this cell while a job is queued.


In [ ]:
job_file = OUT/'job.json'
if SUBMIT_NEW_JOB:
    assert CONNECT_TO_IBM and CONFIRM_OPEN_PLAN and backend is not None
    assert OPEN_INSTANCE_CRN and OPEN_INSTANCE_CRN.startswith('crn:')
    assert ALLOW_RESUBMIT or not job_file.exists(), 'Prior job exists. Retrieve it below; do not submit twice.'
    from qiskit_ibm_runtime import SamplerV2
    sampler = SamplerV2(mode=backend)
    job = sampler.run(isa_circuits, shots=SHOTS)
    OUT.mkdir(parents=True, exist_ok=True)
    job_file.write_text(json.dumps({'job_id':job.job_id(), 'backend':backend.name, 'shots':SHOTS,
                                    'pairs':pairs, 'ideal_fidelities':ideals}, indent=2), encoding='utf-8')
    print('Submitted job ID:', job.job_id(), '| status:', job.status())
else:
    print('Submission skipped. Existing job file:', job_file.exists())

## 6. Retrieve and compare hardware results

Set `CONNECT_TO_IBM=True` to retrieve an existing job after a restart; keep `SUBMIT_NEW_JOB=False`. `job.result()` may wait for the QPU queue. Hardware values are noisy finite-shot estimates; this is a kernel sanity check, **not** evidence that a hardware QSVM outperforms ML.


In [ ]:
if CONNECT_TO_IBM and job_file.exists():
    record = json.loads(job_file.read_text(encoding='utf-8'))
    job = service.job(record['job_id'])
    print('Retrieving job:', record['job_id'], '| status:', job.status())
    result = job.result()
    rows = []
    for pair, ideal, pub in zip(record['pairs'], record['ideal_fidelities'], result):
        counts = pub.data.c.get_counts()
        total = sum(counts.values())
        observed = counts.get('0000', 0) / total
        rows.append({'i':pair[0], 'j':pair[1], 'ideal_fidelity':ideal,
                     'hardware_fidelity':observed, 'shots':total,
                     'shot_standard_error':float(np.sqrt(observed*(1-observed)/total))})
    assert len(rows) == 4
    table = pd.DataFrame(rows)
    OUT.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUT/'comparison.csv', index=False)
    display(table)
    print('Saved comparison; backend:', record['backend'], '| job ID:', record['job_id'])
else:
    print('No hardware result to retrieve. The local checks above are complete.')